In [4]:
# 1. Imports, paths, account detection

import sys
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# --- Make src.* importable from any working directory ---
_root = Path.cwd()
while not (_root / 'src').is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import (
    get_data_path,
    get_base_path,
    get_project_root,
)

# --- Timezone offset (0 = keep UTC) ---
UTC_OFFSET_HOURS = 0

# --- Paths ---
PROJECT_ROOT = get_project_root()
DATA_DIR = get_data_path()
ACCOUNT_DATA_DIR = DATA_DIR / 'Spotify' / 'Spotify_Account_ Data'
BASE_PATH = get_base_path()

# --- Detect account directories ---
# 1) multiple accounts: ACCOUNT_DATA_DIR / <account> / *.json
# 2) single account:    ACCOUNT_DATA_DIR / *.json
json_files = list(ACCOUNT_DATA_DIR.glob('*.json'))
if json_files:
    account_dirs = [ACCOUNT_DATA_DIR]
else:
    account_dirs = [p.parent for p in ACCOUNT_DATA_DIR.glob('*/*.json')]

account_dirs = sorted(set(account_dirs))

if not account_dirs:
    raise FileNotFoundError(f'No JSON files found under {ACCOUNT_DATA_DIR}')

# --- Summary ---
print(f'UTC offset:      {UTC_OFFSET_HOURS}')
print(f'Account dirs:    {len(account_dirs)}')
print(f'UTC offset:      {UTC_OFFSET_HOURS}')
print(f'Account dirs:    {len(account_dirs)}')

UTC offset:      0
Account dirs:    3
UTC offset:      0
Account dirs:    3


In [5]:
# 2. Load YourLibrary.json -> library_tracks.parquet

all_tracks = []

for account_dir in account_dirs:
    json_path = account_dir / 'YourLibrary.json'
    if not json_path.exists():
        continue

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as uf:
                user_data = json.load(uf)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    for item in data.get('tracks', []):
        all_tracks.append({
            'spotify_track_uri': item.get('uri', ''),
            'track_name': item.get('track', ''),
            'artist_name': item.get('artist', ''),
            'album_name': item.get('album', ''),
            'account': account,
        })

df_library = pd.DataFrame(all_tracks)

df_library = df_library[
    df_library['spotify_track_uri'].notna() &
    (df_library['spotify_track_uri'].str.strip() != '')
].copy()

df_library['is_own_library'] = True
df_library['added_at'] = pd.NaT

df_library.to_parquet(BASE_PATH / 'library_tracks.parquet', index=False)

print(f'Library rows:     {len(df_library)}')
print(f'Unique URIs:      {df_library["spotify_track_uri"].nunique()}')
print(f'Accounts:         {df_library["account"].nunique()}')
print(f'Saved:            library_tracks.parquet')

print(df_library.dtypes)

Library rows:     2567
Unique URIs:      2358
Accounts:         3
Saved:            library_tracks.parquet
spotify_track_uri            object
track_name                   object
artist_name                  object
album_name                   object
account                      object
is_own_library                 bool
added_at             datetime64[ns]
dtype: object


In [6]:
# 3. Load Playlist*.json -> playlists_metadata.parquet

EXCLUDED_PLAYLISTS = {'Orphan', 'Liked_Songs'}

def normalize_name(name):
    return name.lower().strip().replace(' ', '_').replace('*', '').replace('"', '').strip('_')

# --- Map account_dir -> spotify_id (username from UserAttributes.json) ---
account_to_spotify_id = {}
for account_dir in account_dirs:
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_attrs = json.load(f)
            username = user_attrs.get('username')
            if username:
                account_to_spotify_id[account_dir] = username
        except Exception:
            pass

playlist_data = []

for account_dir in account_dirs:
    # Account: email/username from UserAttributes, fallback to dir name
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    playlist_files = list(account_dir.glob('Playlist*.json'))
    for json_path in playlist_files:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for pl in data.get('playlists', []):
            name = pl.get('name', '')
            if name in EXCLUDED_PLAYLISTS:
                continue

            last_modified = pl.get('lastModifiedDate', '')
            items = pl.get('items', [])
            num_tracks = len(items)
            collaborators = pl.get('collaborators', [])
            collaborative = bool(collaborators)
            number_of_followers = pl.get('numberOfFollowers', 0)
            owner_id = account_to_spotify_id.get(account_dir, account)

            for item in items:
                track_info = item.get('track', {}) or {}
                playlist_data.append({
                    'account': account,
                    'playlist_name': name,
                    'name_norm': normalize_name(name),
                    'last_modified': last_modified,
                    'num_tracks': num_tracks,
                    'spotify_track_uri': track_info.get('trackUri', ''),
                    'track_name': track_info.get('trackName', ''),
                    'artist_name': track_info.get('artistName', ''),
                    'album_name': track_info.get('albumName', ''),
                    'owner_id': owner_id,
                    'collaborative': collaborative,
                    'collaborators': json.dumps(collaborators, ensure_ascii=False),
                    'number_of_followers': number_of_followers,
                })

df_playlists = pd.DataFrame(playlist_data)

# --- Drop rows with empty URI ---
df_playlists = df_playlists[
    df_playlists['spotify_track_uri'].notna() &
    (df_playlists['spotify_track_uri'].str.strip() != '')
].copy()

# --- Save ---
df_playlists.to_parquet(BASE_PATH / 'playlists_metadata.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_playlists)}')
print(f'Unique playlists:  {df_playlists["playlist_name"].nunique()}')
print(f'Accounts:          {df_playlists["account"].nunique()}')
print(f'Saved:             playlists_metadata.parquet')

print(df_playlists.dtypes)

Rows:              13417
Unique playlists:  157
Accounts:          3
Saved:             playlists_metadata.parquet
account                object
playlist_name          object
name_norm              object
last_modified          object
num_tracks              int64
spotify_track_uri      object
track_name             object
artist_name            object
album_name             object
owner_id               object
collaborative            bool
collaborators          object
number_of_followers     int64
dtype: object


In [7]:
# 4. Load UserAttributes.json -> user_attributes.parquet

user_info = []

for account_dir in account_dirs:
    user_attrs_path = account_dir / 'UserAttributes.json'
    if not user_attrs_path.exists():
        continue

    try:
        with open(user_attrs_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    account = data.get('email') or data.get('username') or account_dir.name

    user_info.append({
        'account': account,
        'email': data.get('email', ''),
        'country': data.get('country', ''),
        'language': data.get('language', ''),
        'birthdate': data.get('birthdate', ''),
        'gender': data.get('gender', ''),
    })

df_user = pd.DataFrame(user_info)
df_user.to_parquet(BASE_PATH / 'user_attributes.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_user)}')
print(f'Columns:           {list(df_user.columns)}')
print(f'Saved:             user_attributes.parquet')

print(df_user.dtypes)

Rows:              3
Columns:           ['account', 'email', 'country', 'language', 'birthdate', 'gender']
Saved:             user_attributes.parquet
account      object
email        object
country      object
language     object
birthdate    object
gender       object
dtype: object


In [8]:
# 5. Load Inferences.json -> inferences.parquet

inference_list = []

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    json_path = account_dir / 'Inferences.json'
    if not json_path.exists():
        continue

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    for value in data.get('inferences', []):
        inference_list.append({
            'account': account,
            'value': value,
        })

df_inf = pd.DataFrame(inference_list)
df_inf.to_parquet(BASE_PATH / 'inferences.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_inf)}')
print(f'Accounts:          {df_inf["account"].nunique() if len(df_inf) else 0}')
print(f'Saved:             inferences.parquet')

print(df_inf.dtypes)

Rows:              104
Accounts:          2
Saved:             inferences.parquet
account    object
value      object
dtype: object


In [9]:
# 6. Load SearchQueries.json -> search_queries.parquet

search_list = []

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    json_path = account_dir / 'SearchQueries.json'
    if not json_path.exists():
        continue

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    if isinstance(data, list):
        queries = data
    else:
        queries = data.get('searchQueries', [])

    for q in queries:
        if isinstance(q, dict):
            search_list.append({
                'account': account,
                'query': q.get('searchQuery', ''),
                'date': q.get('searchInteractionDate', ''),
            })
        else:
            search_list.append({
                'account': account,
                'query': q,
                'date': None,
            })

df_search = pd.DataFrame(search_list)
df_search.to_parquet(BASE_PATH / 'search_queries.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_search)}')
print(f'Accounts:          {df_search["account"].nunique() if len(df_search) else 0}')
print(f'Saved:             search_queries.parquet')

print(df_search.dtypes)

Rows:              13
Accounts:          1
Saved:             search_queries.parquet
account    object
query      object
date       object
dtype: object


In [10]:
# 7. Build owner_ids.parquet

rows = []

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    identity_path = account_dir / 'Identity.json'

    username = None
    display_name = account

    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_attrs = json.load(f)
            username = user_attrs.get('username')
        except Exception:
            pass

    if identity_path.exists():
        try:
            with open(identity_path, 'r', encoding='utf-8') as f:
                identity = json.load(f)
            display_name = identity.get('displayName', display_name)
        except Exception:
            pass

    if username:
        rows.append({
            'owner_id': username,
            'display_name': display_name,
        })

owner_ids_df = pd.DataFrame(rows).drop_duplicates('owner_id')
owner_ids_df.to_parquet(BASE_PATH / 'owner_ids.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(owner_ids_df)}')
print(f'Columns:           {list(owner_ids_df.columns)}')
print(f'Saved:             owner_ids.parquet')

print(owner_ids_df.dtypes)

Rows:              3
Columns:           ['owner_id', 'display_name']
Saved:             owner_ids.parquet
owner_id        object
display_name    object
dtype: object


In [11]:
# 8. Classify playlists in playlist_links.parquet

playlist_links = pd.read_parquet(BASE_PATH / 'playlist_links.parquet')
playlists_metadata = pd.read_parquet(BASE_PATH / 'playlists_metadata.parquet')
owner_ids = set(pd.read_parquet(BASE_PATH / 'owner_ids.parquet')['owner_id'].tolist())

# --- Clean empty URIs ---
playlist_links = playlist_links[
    playlist_links['Track URI'].notna() &
    (playlist_links['Track URI'].str.strip() != '')
].copy()

# --- Add name_norm ---
playlist_links['name_norm'] = playlist_links['playlist_name'].apply(normalize_name)

if 'name_norm' not in playlists_metadata.columns:
    playlists_metadata['name_norm'] = playlists_metadata['playlist_name'].apply(normalize_name)

# --- Drop technical duplicates ---
before = len(playlist_links)
playlist_links = playlist_links.drop_duplicates(
    subset=['name_norm', 'Track URI', 'Added By', 'Added At'],
    keep='first'
)
dropped_dup = before - len(playlist_links)

# --- Personal / collaborative sets ---
personal_names = set(playlists_metadata[~playlists_metadata['collaborative']]['name_norm'].unique())
collaborative_names = set(playlists_metadata[playlists_metadata['collaborative']]['name_norm'].unique())

# --- Adders per playlist ---
adders_map = playlist_links.groupby('name_norm')['Added By'].apply(set).to_dict()

def classify_playlist(name_norm):
    if name_norm in personal_names:
        return 'personal'
    elif name_norm in collaborative_names:
        return 'collaborative'
    else:
        adders = adders_map.get(name_norm, set())
        if any(ad in owner_ids for ad in adders if pd.notna(ad)):
            return 'contributed'
        else:
            return 'foreign'

playlist_links['playlist_type'] = playlist_links['name_norm'].apply(classify_playlist)

def has_external_adders(name_norm):
    adders = adders_map.get(name_norm, set())
    for ad in adders:
        if pd.isna(ad):
            continue
        if ad == 'unknown':
            continue
        if ad not in owner_ids:
            return True
    return False

playlist_links['has_external_adders'] = playlist_links['name_norm'].apply(has_external_adders)

# --- Save ---
playlist_links.to_parquet(BASE_PATH / 'playlist_links.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(playlist_links)}')
print(f'Dropped (dupes):   {dropped_dup}')
print(f'Saved:             playlist_links.parquet')
print(f'\nplaylist_type:')
print(playlist_links['playlist_type'].value_counts().to_string())

print(playlist_links.dtypes)

Rows:              33094
Dropped (dupes):   7553
Saved:             playlist_links.parquet

playlist_type:
playlist_type
foreign          19534
personal         10028
collaborative     3443
contributed         89
Track URI              object
Added By               object
Added At               object
playlist_name          object
account                object
name_norm              object
playlist_type          object
has_external_adders      bool
dtype: object


In [12]:
# 9. Build track_dates.parquet

EXPORTIFY_DIR = DATA_DIR / 'Exportify'

# --- 1. Library-added dates from Liked_Songs.csv ---
liked_csv_files = list(EXPORTIFY_DIR.rglob('Liked_Songs.csv'))
if liked_csv_files:
    liked_dfs = []
    for f in liked_csv_files:
        df_liked = pd.read_csv(f)
        liked_dfs.append(df_liked[['Track URI', 'Added At']])
    liked_dates_all = pd.concat(liked_dfs, ignore_index=True)
    liked_dates_all['Added At'] = (
        pd.to_datetime(liked_dates_all['Added At'], errors='coerce')
        + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    )
    liked_dates_all = liked_dates_all.dropna(subset=['Added At'])
    library_added_dates = (
        liked_dates_all.groupby('Track URI')['Added At'].min().reset_index()
    )
    library_added_dates.columns = ['spotify_track_uri', 'library_added_at']
else:
    library_added_dates = pd.DataFrame(columns=['spotify_track_uri', 'library_added_at'])

# --- 2. First-added-to-own-playlist dates ---
pl = pd.read_parquet(BASE_PATH / 'playlist_links.parquet')
OWNER_IDS = set(pd.read_parquet(BASE_PATH / 'owner_ids.parquet')['owner_id'].tolist())

my_pl = pl[pl['Added By'].isin(OWNER_IDS)]
EXCLUDED_PLAYLISTS_NORM = {'liked_songs', 'orphan'}
my_pl = my_pl[~my_pl['name_norm'].isin(EXCLUDED_PLAYLISTS_NORM)]

playlist_added_dates = (
    my_pl.groupby('Track URI')['Added At'].min().reset_index()
)
playlist_added_dates['Added At'] = (
    pd.to_datetime(playlist_added_dates['Added At'], errors='coerce')
    + pd.Timedelta(hours=UTC_OFFSET_HOURS)
)
playlist_added_dates = playlist_added_dates.dropna(subset=['Added At'])
playlist_added_dates.columns = ['spotify_track_uri', 'first_playlist_added_at']

# --- 3. Merge both date sources ---
track_dates = pd.merge(
    library_added_dates, playlist_added_dates,
    on='spotify_track_uri', how='outer'
)

# --- Save ---
track_dates.to_parquet(BASE_PATH / 'track_dates.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(track_dates)}')
print(f'With library date: {int(track_dates["library_added_at"].notna().sum())}')
print(f'With playlist date:{int(track_dates["first_playlist_added_at"].notna().sum())}')
print(f'Both dates:        {int((track_dates["library_added_at"].notna() & track_dates["first_playlist_added_at"].notna()).sum())}')
print(f'Saved:             track_dates.parquet')

print(track_dates.dtypes)

Rows:              10315
With library date: 2358
With playlist date:9641
Both dates:        1684
Saved:             track_dates.parquet
spotify_track_uri                       object
library_added_at           datetime64[ns, UTC]
first_playlist_added_at    datetime64[ns, UTC]
dtype: object


## Содержание `00_2_account_data.ipynb`

**Цель ноутбука:**
Загрузить данные аккаунтов из выгрузки Spotify Account Data, создать справочные артефакты: библиотеку, метаданные плейлистов, демографию, инференсы, поисковые запросы, `owner_ids`. Затем классифицировать плейлисты в `playlist_links` и собрать `track_dates`.

**Общие принципы:**
- Пути задаются в ячейке 1 через `src.paths`, без хардкода.
- Аккаунты определяются динамически через `account_dirs`: подпапки = аккаунты, либо корень = один аккаунт.
- В выводе ячеек — только счётчики и `df.dtypes`. Никаких таблиц, имён аккаунтов и путей.
- Каждая ячейка читает/создаёт конкретный Parquet-файл.
- Приватные паркеты (внутри `data/00_output/`) остаются как есть, маскировка делается на границе с публичным.

---

### Ячейка 1. Импорты, пути, определение аккаунтов
**Что делает:**
- Импортирует библиотеки, добавляет корень репо в `sys.path`.
- Загружает пути через `src.paths` (`get_project_root`, `get_data_path`, `get_base_path`).
- Определяет `ACCOUNT_DATA_DIR` и `account_dirs`:
  - если JSON лежат в подпапках → каждая подпапка = аккаунт;
  - если JSON в корне → корень = один аккаунт.
- Сортирует, убирает дубли.
- `UTC_OFFSET_HOURS = 0` — смещение локального времени от UTC.

**Артефакты:** нет.

---

### Ячейка 2. `YourLibrary.json` → `library_tracks.parquet`
**Вход:** `YourLibrary.json` в каждой директории аккаунта.
**Выход:** `library_tracks.parquet`.

**Логика:**
- Для каждого `account_dir` находит `YourLibrary.json` (пропускает, если нет).
- Определяет `account` из `UserAttributes.json` (email/username), fallback — имя директории.
- Извлекает треки (`uri`, `track`, `artist`, `album`).
- Удаляет строки с пустым `spotify_track_uri`.
- **Дубликаты по URI сохраняет** — лайки на разных аккаунтах.
- Служебные поля: `is_own_library=True`, `added_at=NaT`.
- Сохраняет файл.

---

### Ячейка 3. `Playlist*.json` → `playlists_metadata.parquet`
**Вход:** `Playlist*.json` в каждой директории аккаунта.
**Выход:** `playlists_metadata.parquet`.

**Логика:**
- Для каждого `account_dir` находит все `Playlist*.json`.
- Определяет `account` и `owner_id` (`username` из `UserAttributes.json`).
- Исключает технические плейлисты `Liked_Songs` и `Orphan`.
- Для каждого плейлиста нормализует имя, фиксирует `last_modified`, `num_tracks`, `collaborative`, `collaborators`, `number_of_followers`.
- Разворачивает `items` в строки.
- Удаляет строки с пустым `spotify_track_uri`.
- **Дубликаты сохраняет** (повторные добавления треков — поведенческий сигнал).
- Сохраняет файл.

---

### Ячейка 4. `UserAttributes.json` → `user_attributes.parquet`
**Вход:** `UserAttributes.json` в каждой директории аккаунта.
**Выход:** `user_attributes.parquet`.

**Логика:**
- Для каждого `account_dir` читает `UserAttributes.json` (пропускает, если нет).
- Определяет `account` из `email`/`username`, fallback — имя директории.
- Извлекает `email`, `country`, `language`, `birthdate`, `gender`.
- Сохраняет файл.

---

### Ячейка 5. `Inferences.json` → `inferences.parquet`
**Вход:** `Inferences.json` в каждой директории аккаунта.
**Выход:** `inferences.parquet`.

**Логика:**
- Для каждого `account_dir` читает `Inferences.json` (пропускает, если нет).
- Определяет `account`.
- Извлекает все значения из массива `inferences`.
- Каждое предположение — отдельная строка (`account`, `value`).
- Сохраняет файл.

**Примечание:** файл есть только у двух аккаунтов.

---

### Ячейка 6. `SearchQueries.json` → `search_queries.parquet`
**Вход:** `SearchQueries.json` в каждой директории аккаунта.
**Выход:** `search_queries.parquet`.

**Логика:**
- Для каждого `account_dir` читает `SearchQueries.json` (пропускает, если нет).
- Поддерживает два формата: список объектов или объект с ключом `searchQueries`.
- Извлекает `searchQuery` и `searchInteractionDate`.
- Сохраняет файл.

**Примечание:** файл есть только у одного аккаунта.

---

### Ячейка 7. Создание `owner_ids.parquet`
**Вход:** `UserAttributes.json`, `Identity.json` в каждой директории аккаунта.
**Выход:** `owner_ids.parquet`.

**Логика:**
- Для каждого `account_dir` читает `username` из `UserAttributes.json`.
- Если есть `Identity.json` — берёт `displayName`, иначе fallback на имя директории.
- Формирует строки `owner_id` и `display_name`.
- Если `username` отсутствует — аккаунт пропускается.
- Удаляет дубликаты по `owner_id`.
- Сохраняет файл.

---

### Ячейка 8. Классификация плейлистов в `playlist_links.parquet`
**Вход:** `playlist_links.parquet`, `playlists_metadata.parquet`, `owner_ids.parquet`.
**Выход:** обновлённый `playlist_links.parquet`.

**Логика:**
- Загружает `playlist_links`, добавляет `name_norm` через `normalize_name`.
- Очищает от пустых `Track URI`.
- Строит множества личных (`personal`) и совместных (`collaborative`) плейлистов.
- Удаляет **технические дубликаты** — одинаковые `name_norm`, `Track URI`, `Added By`, `Added At`.
- Для каждого плейлиста строит множество `Added By`.
- Классифицирует тип:
  - `personal` — есть в personal-множестве;
  - `collaborative` — есть в collaborative-множестве;
  - `contributed` — среди добавщиков есть хотя бы один `owner_id`;
  - `foreign` — все остальные.
- Определяет `has_external_adders`: `True`, если есть добавщик не из `owner_ids` и не `'unknown'`.
- Сохраняет файл.

---

### Ячейка 9. Даты добавления → `track_dates.parquet`
**Вход:** `Liked_Songs.csv`, `playlist_links.parquet`, `owner_ids.parquet`.
**Выход:** `track_dates.parquet`.

**Логика:**
- Загружает все `Liked_Songs.csv`, берёт `Track URI` и `Added At`.
- Минимальная дата для каждого URI → `library_added_at`.
- Фильтрует `playlist_links` по `Added By in owner_ids`.
- Исключает технические плейлисты (`liked_songs`, `orphan`).
- Минимальная дата добавления в свои плейлисты → `first_playlist_added_at`.
- Объединяет даты лайков и плейлистов по `spotify_track_uri` (outer join).
- Сохраняет файл.

---

### Итог

После выполнения всех ячеек `00_2_account_data.ipynb` в `data/00_output/` появляются или обновляются:

- `library_tracks.parquet`
- `playlists_metadata.parquet`
- `user_attributes.parquet`
- `inferences.parquet`
- `search_queries.parquet`
- `owner_ids.parquet`
- `playlist_links.parquet` (с `name_norm`, `playlist_type`, `has_external_adders`)
- `track_dates.parquet`

Все файлы приватные, в git не попадают. Публичные агрегаты для дашборда собираются отдельно в `03_dashboard/dashboard/datasets/`.